# Phase 2 — PAN Train/Val/Test Splits

Creates **stratified, document-leakage-free** splits on `master.csv`.

### Why document-level splitting?
If we split rows randomly, the same `suspicious_id` could land in both train and test — the model would 'cheat' by memorizing document-specific patterns. We split on `suspicious_id` so every document and all its passages stay in one partition.

### Output
All saved into `data/processed/` alongside your existing files:
- `pan_train.csv` — 70%
- `pan_val.csv`   — 15%
- `pan_test.csv`  — 15%

Stratification preserves the ratio of `(corpus_type, is_plagiarism)` combinations across splits.

In [4]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import StratifiedShuffleSplit

# Mount Drive (where master.csv was saved during Phase 1)
from google.colab import drive
drive.mount('/content/drive')

RANDOM_STATE = 42

PROCESSED_DIR = Path('/content/drive/MyDrive/pan-processed')
INPUT_CSV     = PROCESSED_DIR / 'master.csv'

df = pd.read_csv(INPUT_CSV)
print(f'Loaded master.csv: {df.shape}')
print(f'Unique suspicious documents: {df["suspicious_id"].nunique():,}')

Mounted at /content/drive
Loaded master.csv: (76661, 16)
Unique suspicious documents: 11,093


## Strategy

We need both:
1. **Group safety** — same `suspicious_id` cannot appear in two splits
2. **Stratification** — preserve `(corpus_type, is_plagiarism)` ratio

The trick: for each document, find its **dominant stratum** (most common combination of corpus_type + is_plagiarism), then do a stratified split on documents using that label. This gives both properties at once.

In [5]:
# Build per-document stratification key
doc_labels = (
    df.groupby('suspicious_id')
      .agg(corpus_type=('corpus_type', 'first'),
           dominant_label=('is_plagiarism', lambda x: int(x.mean() >= 0.5)))
      .reset_index()
)
doc_labels['stratum'] = doc_labels['corpus_type'] + '_' + doc_labels['dominant_label'].astype(str)
print('Document-level stratum distribution')
print(doc_labels['stratum'].value_counts())

Document-level stratum distribution
stratum
external_1    6501
external_0    4592
Name: count, dtype: int64


In [6]:
# Stage 1: 85% train+val, 15% test
sss1 = StratifiedShuffleSplit(n_splits=1, test_size=0.15, random_state=RANDOM_STATE)
trainval_idx, test_idx = next(sss1.split(doc_labels, doc_labels['stratum']))
trainval_docs = doc_labels.iloc[trainval_idx]
test_docs     = doc_labels.iloc[test_idx]

# Stage 2: from 85%, split into ~70% train + ~15% val (15/85 = 0.176)
sss2 = StratifiedShuffleSplit(n_splits=1, test_size=0.176, random_state=RANDOM_STATE)
train_idx, val_idx = next(sss2.split(trainval_docs, trainval_docs['stratum']))
train_docs = trainval_docs.iloc[train_idx]
val_docs   = trainval_docs.iloc[val_idx]

print(f'Train docs: {len(train_docs):,}')
print(f'Val docs  : {len(val_docs):,}')
print(f'Test docs : {len(test_docs):,}')

Train docs: 7,769
Val docs  : 1,660
Test docs : 1,664


In [7]:
# Map back to row-level splits and verify zero leakage
train_ids = set(train_docs['suspicious_id'])
val_ids   = set(val_docs['suspicious_id'])
test_ids  = set(test_docs['suspicious_id'])

assert train_ids & val_ids  == set()
assert train_ids & test_ids == set()
assert val_ids   & test_ids == set()
print('Zero document leakage confirmed')

df_train = df[df['suspicious_id'].isin(train_ids)].reset_index(drop=True)
df_val   = df[df['suspicious_id'].isin(val_ids)].reset_index(drop=True)
df_test  = df[df['suspicious_id'].isin(test_ids)].reset_index(drop=True)

print(f'\nRow counts:')
print(f'  Train: {len(df_train):,}')
print(f'  Val  : {len(df_val):,}')
print(f'  Test : {len(df_test):,}')

Zero document leakage confirmed

Row counts:
  Train: 54,229
  Val  : 10,776
  Test : 11,656


In [8]:
def split_summary(name, df_split):
    print(f'\n{name} ({len(df_split):,} rows)')
    vc = df_split['is_plagiarism'].value_counts(normalize=True).round(3)
    print(f'  Positive: {vc.get(1,0):.1%}')
    print(f'  Negative: {vc.get(0,0):.1%}')
    print('  By corpus_type:')
    print(df_split['corpus_type'].value_counts().to_string())
    print('  By plagiarism_type:')
    print(df_split['plagiarism_type'].value_counts().to_string())

split_summary('TRAIN', df_train)
split_summary('VAL',   df_val)
split_summary('TEST',  df_test)


TRAIN (54,229 rows)
  Positive: 79.8%
  Negative: 20.2%
  By corpus_type:
corpus_type
external     42811
intrinsic    11418
  By plagiarism_type:
plagiarism_type
artificial     35291
none           10938
translation     4573
simulated       3427

VAL (10,776 rows)
  Positive: 78.4%
  Negative: 21.6%
  By corpus_type:
corpus_type
external     8378
intrinsic    2398
  By plagiarism_type:
plagiarism_type
artificial     7308
none           2323
translation    1030
simulated       115

TEST (11,656 rows)
  Positive: 80.0%
  Negative: 20.0%
  By corpus_type:
corpus_type
external     9276
intrinsic    2380
  By plagiarism_type:
plagiarism_type
artificial     7202
none           2337
simulated      1066
translation    1051


In [9]:
# Save splits into the same Drive folder
df_train.to_csv(PROCESSED_DIR / 'pan_train.csv', index=False)
df_val.to_csv(PROCESSED_DIR / 'pan_val.csv',     index=False)
df_test.to_csv(PROCESSED_DIR / 'pan_test.csv',   index=False)

print('Saved to Drive:')
for f in sorted(PROCESSED_DIR.glob('pan_*.csv')):
    size = f.stat().st_size / 1_048_576
    print(f'  {f.name}  ({size:.1f} MB)')

Saved to Drive:
  pan_test.csv  (65.2 MB)
  pan_train.csv  (317.2 MB)
  pan_val.csv  (63.0 MB)
